# Finance Guidance Assistant — RAG Prototype (Colab)

Run all cells top to bottom. Make sure a GPU runtime is selected:
**Runtime → Change runtime type → Hardware accelerator → GPU (T4)**

This notebook:
1. Installs dependencies
2. Gets the project code (clone from GitHub, or upload manually)
3. Builds the FAISS index from the finance knowledge base
4. Runs a quick retrieval sanity check
5. Launches the Gradio app (shareable public link)
6. Runs the evaluation suite and shows results


## 1. Install dependencies

In [ ]:
!pip install sentence-transformers faiss-cpu transformers torch gradio accelerate --quiet

## 2. Get the project code

**Option A — clone from GitHub** (once you've pushed the repo, replace the URL below):


In [ ]:
# Replace with your actual repo URL once pushed
GITHUB_REPO_URL = "https://github.com/YOUR-USERNAME/finance-rag.git"

!git clone {GITHUB_REPO_URL}
%cd finance-rag

**Option B — no repo yet?** Upload the `finance-rag` folder as a zip instead:
1. Zip the `finance-rag` folder on your machine
2. Run the cell below, click "Choose Files", select the zip


In [ ]:
# Only run this cell if you're using Option B (manual upload) instead of git clone
from google.colab import files
import zipfile

uploaded = files.upload()  # select your finance-rag.zip
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall('.')

%cd finance-rag

## 3. Build the FAISS index

In [ ]:
import sys
sys.path.insert(0, 'src')

from chunking import build_chunk_corpus
from embed_index import build_index, save_index

corpus = build_chunk_corpus('data/finance_kb.json')
print(f"Loaded {len(corpus)} chunks from knowledge base.")

index, model, corpus = build_index(corpus)
save_index(index, corpus, 'data/faiss_index.bin', 'data/corpus.json')
print(f"Built FAISS index with {index.ntotal} vectors (dim={index.d}).")

## 4. Quick retrieval sanity check

In [ ]:
from retrieve import retrieve

test_queries = [
    "When can I take money out of my pension?",
    "What's the difference between saving and investing?",
    "How do I improve my credit score?",
]

for q in test_queries:
    print(f"\nQuery: {q}")
    results = retrieve(q, index, model, corpus, top_k=3)
    for r in results:
        print(f"  [{r['score']:.3f}] {r['title']} ({r['category']})")

## 5. Test the safeguard

In [ ]:
from safeguard import check_safeguards

test_cases = [
    "Should I transfer my pension into a SIPP given my situation?",
    "What is a Lifetime ISA?",
]

for q in test_cases:
    retrieved = retrieve(q, index, model, corpus, top_k=3)
    blocked, message = check_safeguards(q, retrieved)
    print(f"Query: {q}")
    print(f"  Blocked: {blocked}")
    if blocked:
        print(f"  Message: {message}")
    print()

## 6. Load the generation model and test end-to-end

In [ ]:
from generate import Generator, format_answer_with_citations

generator = Generator()  # downloads Qwen2.5-1.5B-Instruct on first run

query = "What is a Lifetime ISA?"
retrieved = retrieve(query, index, model, corpus, top_k=3)
blocked, message = check_safeguards(query, retrieved)

if blocked:
    print(message)
else:
    answer = generator.generate(query, retrieved)
    print(format_answer_with_citations(answer, retrieved))

## 7. Launch the Gradio app

In [ ]:
import gradio as gr

def answer_question(query, history=None):
    if not query or not query.strip():
        return "Please enter a question."
    retrieved = retrieve(query, index, model, corpus, top_k=3)
    blocked, message = check_safeguards(query, retrieved)
    if blocked:
        return message
    raw_answer = generator.generate(query, retrieved)
    return format_answer_with_citations(raw_answer, retrieved)

demo = gr.ChatInterface(
    fn=answer_question,
    title="Finance Guidance Assistant",
    description="Ask general questions about UK pensions, savings, debt, mortgages, credit, and investing. This is guidance, not regulated financial advice.",
    examples=[
        "What is the difference between a defined contribution and defined benefit pension?",
        "What is a Lifetime ISA?",
        "How can I recognise a pension scam?",
        "Should I move my pension into a SIPP given my situation?",
    ],
)

demo.launch(share=True)  # take a screenshot of the public link + a sample conversation for your report

## 8. Run the evaluation suite

Stop the Gradio cell above first (interrupt it), then run this.


In [ ]:
%cd ..
!python eval/evaluate.py
%cd finance-rag

## 9. View evaluation results

These numbers go straight into your Part B technical report's evaluation section.


In [ ]:
import pandas as pd

retrieval_df = pd.read_csv('eval/retrieval_results.csv')
print("Retrieval accuracy:", retrieval_df['retrieval_hit'].mean())
print("Average retrieval latency (s):", retrieval_df['retrieval_latency_sec'].mean())
retrieval_df

In [ ]:
generation_df = pd.read_csv('eval/generation_results.csv')
generation_df
# Fill in the 'manual_grade' column (correct / partial / incorrect) by reading each
# generated_answer against the ground_truth, then re-load this CSV to compute
# an overall accuracy percentage for your report.

In [ ]:
safeguard_df = pd.read_csv('eval/safeguard_results.csv')
safeguard_df